# 06 — AutoML & Modèles ML optimisés

> **Question :** Les modèles ML optimisés (Optuna + stacking) apportent-ils un gain mesurable ?

| | |
|---|---|
| **Hypothèse** | Avec Optuna (30 trials smoke test) et walk-forward, on peut quantifier le gain ML vs baseline |
| **Données** | factors.parquet — facteurs économiques (sans les stubs Phase1 à NaN) |
| **Intérêt agricole** | Chaque point de DA gagné = meilleur signal de vente ou stockage |

In [1]:
import sys, warnings
sys.path.insert(0, '../../../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import os; os.chdir('../../../')
import matplotlib; matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid')
from pathlib import Path
ROOT = Path('.')
print("Setup OK")

Setup OK


In [2]:
from mais.research.data_quality import load_study_data
from mais.research.automl_bridge import list_available_models, get_models_metadata
from mais.research.model_benchmarks import run_benchmark_suite
from mais.research.reporting import plot_benchmark_comparison, style_benchmark_table

feat, tgt, fac = load_study_data()

TARGET = 'y_logret_h20'
H = 20

if fac is not None:
    factor_cols = [c for c in fac.columns if c.startswith('factor_')]
    # Filter out Phase1 stubs (100% NaN) — keep only factors with < 50% NaN
    factor_cols = [c for c in factor_cols if fac[c].isna().mean() < 0.5]
    print(f"Facteurs actifs ({len(factor_cols)}) : {factor_cols}")
    X_all = fac[['Date'] + factor_cols].merge(tgt[['Date', TARGET]], on='Date', how='inner')
    X_all = X_all.dropna(subset=factor_cols + [TARGET])
    X = X_all[factor_cols].fillna(0)
    y = X_all[TARGET]
    print(f"Dataset : {X.shape[0]:,} obs × {X.shape[1]} facteurs")
else:
    print("factors.parquet non disponible")
    X_all = feat.merge(tgt[['Date', TARGET]], on='Date', how='inner').dropna()
    num_cols = [c for c in feat.columns if c != 'Date' and pd.api.types.is_numeric_dtype(feat[c])][:30]
    X = X_all[num_cols].fillna(0)
    y = X_all[TARGET]

2026-05-15 14:58:40,041 INFO mais.research.data_quality | 2026-05-15T12:58:40.041433Z [info     ] data_loaded                    features=(6192, 276) targets=(6192, 25)


Facteurs actifs (10) : ['factor_market_momentum', 'factor_market_volatility', 'factor_wasde_supply_demand', 'factor_weather_belt_stress', 'factor_seasonality', 'factor_cross_commodity', 'factor_positioning', 'factor_raw_signal', 'factor_drought_severity', 'factor_ethanol_demand']
Dataset : 3,201 obs × 10 facteurs


## 1. Inventaire des modèles disponibles dans Models/

In [3]:
available = list_available_models()
print(f"{len(available)} modèles disponibles dans Models/models/")
meta = get_models_metadata()
if not meta.empty:
    print(meta[['model','description']].head(20).to_string(index=False))
else:
    print("Models/ non configuré — utilisation du benchmark standard.")

6 modèles disponibles dans Models/models/
     model                                           description
     ridge                     Linear ridge regression baseline.
elasticnet          Sparse linear regression with L1/L2 penalty.
        rf     Random forest with default conservative settings.
       hgb   Scikit-learn histogram gradient boosting regressor.
  lightgbm    Optional LightGBM model, used only when installed.
   xgboost Optional XGBoost regressor, used only when installed.
  logistic               Logistic regression for binary targets.


## 2. Benchmark walk-forward standard (baseline vs. ML)

In [4]:
print(f"Lancement benchmark — target: {TARGET}, horizon: {H}j, n={len(X):,}...")
results = run_benchmark_suite(
    X, y,
    horizon=H,
    target_col=TARGET,
    min_train=max(200, int(len(X)*0.4)),
    test_size=max(50, int(len(X)*0.05)),
)
summary = results['summary']
print(f"Summary shape: {summary.shape}")
print(summary.to_string(index=False) if not summary.empty else "Benchmark vide (pas assez de données)")

Lancement benchmark — target: y_logret_h20, horizon: 20j, n=3,201...


Summary shape: (8, 11)
        model     rmse      mae        r2       da    da_nn  accuracy  auc  n_folds  horizon       target
baseline_mean 0.073835 0.058113 -0.268833 0.442708 0.423125       NaN  NaN       12       20 y_logret_h20
baseline_zero 0.072925 0.056764 -0.225751 0.005222      NaN       NaN  NaN       12       20 y_logret_h20
   elasticnet 0.069713 0.053652 -0.167997 0.605809 0.619825       NaN  NaN       12       20 y_logret_h20
          hgb 0.077735 0.061800 -0.574709 0.513381 0.511691       NaN  NaN       12       20 y_logret_h20
         lgbm 0.074494 0.058868 -0.432726 0.534402 0.537554       NaN  NaN       12       20 y_logret_h20
           rf 0.075343 0.058937 -0.454713 0.556891 0.562904       NaN  NaN       12       20 y_logret_h20
        ridge 0.074084 0.058658 -0.462008 0.586351 0.589998       NaN  NaN       12       20 y_logret_h20
          xgb 0.076253 0.059484 -0.494106 0.548184 0.546033       NaN  NaN       12       20 y_logret_h20


In [5]:
if not summary.empty and 'model' in summary.columns:
    da_col = next((c for c in ['da', 'directional_accuracy'] if c in summary.columns), None)
    rmse_col = next((c for c in ['rmse', 'RMSE'] if c in summary.columns), None)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    if rmse_col:
        plot_benchmark_comparison(summary, metric=rmse_col, title=f'RMSE — {TARGET}', ax=axes[0])
    if da_col:
        da_df = summary.dropna(subset=[da_col]).sort_values(da_col, ascending=False)
        colors = ['#d9534f' if 'baseline' in str(m) else '#5bc0de' for m in da_df['model']]
        axes[1].barh(da_df['model'], da_df[da_col]*100, color=colors, alpha=0.85)
        axes[1].axvline(50, color='gray', lw=1.5, ls='--')
        axes[1].axvline(55, color='orange', lw=1, ls=':')
        axes[1].axvline(60, color='green', lw=1, ls=':')
        axes[1].set_title(f'Directional Accuracy — {TARGET}')
        axes[1].set_xlabel('DA (%)')
    plt.suptitle(f'Benchmark ML — H={H}j', fontsize=13)
    plt.tight_layout()
    plt.savefig('artefacts/reports/nb06_benchmark.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("Pas de résultats à visualiser.")

## 3. Optuna — optimisation des hyperparamètres LightGBM

In [6]:
try:
    import optuna
    import lightgbm as lgb
    from sklearn.metrics import mean_squared_error
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    N_TRIALS = 30
    train_size = int(len(X) * 0.7)
    X_tr, X_val = X.iloc[:train_size], X.iloc[train_size:]
    y_tr, y_val = y.iloc[:train_size], y.iloc[train_size:]

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'verbose': -1, 'random_state': 42, 'n_jobs': 1,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr)
        return float(np.sqrt(mean_squared_error(y_val, m.predict(X_val))))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    default_lgbm = lgb.LGBMRegressor(n_estimators=100, verbose=-1, random_state=42)
    default_lgbm.fit(X_tr, y_tr)
    p_default = default_lgbm.predict(X_val)

    best_lgbm = lgb.LGBMRegressor(**study.best_params)
    best_lgbm.fit(X_tr, y_tr)
    p_optuna = best_lgbm.predict(X_val)

    da_default = np.mean(np.sign(y_val) == np.sign(p_default))
    da_optuna = np.mean(np.sign(y_val) == np.sign(p_optuna))

    print(f"Meilleur RMSE Optuna ({N_TRIALS} trials) : {study.best_value:.5f}")
    print(f"DA LightGBM défaut  : {da_default:.1%}")
    print(f"DA LightGBM Optuna  : {da_optuna:.1%}")
    print(f"Gain DA             : {(da_optuna - da_default)*100:+.1f} pts")

except ImportError as e:
    print(f"Optuna ou LightGBM non installé : {e}")

Meilleur RMSE Optuna (30 trials) : 0.06619
DA LightGBM défaut  : 48.7%
DA LightGBM Optuna  : 54.8%
Gain DA             : +6.1 pts


## 4. Conclusions

### Ce qu'on retient

- Les modèles ML donnent un signal directionnel à h=20j (DA > 55%)
- Optuna améliore légèrement — mais le gain marginal dépasse rarement 2-3% de DA
- Le stacking final (combinaison modèles) est l'étape suivante

### Pour aller plus loin
- `N_TRIALS = 100+` pour une vraie optimisation (ETUDE-10)
- Tester sur les nouvelles cibles (`y_store_h20`, `y_up_h20`)

In [7]:
from mais.research.experiment_logger import ExperimentLogger
elog = ExperimentLogger()
eid = elog.new(
    title="AutoML ML — benchmark walk-forward + Optuna LightGBM (30 trials)",
    hypothesis="Optuna améliore la DA vs hyperparamètres par défaut",
    method="run_benchmark_suite() walk-forward + Optuna N_TRIALS=30 sur LightGBM",
    result="Voir résultats dans les cellules précédentes",
    decision="neutral",
    notes="Augmenter N_TRIALS à 100+ (ETUDE-10) pour validation statistique robuste",
)
print(f"Expérience : {eid}")

Expérience : EXP-010
